In [1]:
import numpy as np
import pandas as pd
import sys
import os
from IPython.display import display
from tqdm.auto import tqdm
tqdm.pandas()

sys.path.append(os.path.abspath("../../../"))
from EPF import variables

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

c:\Users\Daniel\venv-forecasting\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def heuristic_price_prediction(price_df,operating_protocol_transform, operating_protocol_window_size, apply_blend):

    

    def _apply_transform(operating_protocol_transform):
        
        
        # Mode 1) Actual prices
        # shift(1) so each seasonal group's signal is derived from the previous seasonal group's mean.
        # The NaN row produced by shift(1) falls in the extra leading 14-day buffer and is discarded by _filter().
        if operating_protocol_transform == "None":
            df = pd.pivot_table(price_df,values='Price',index=["Year", "Seasonal_group"],columns="Interval",aggfunc="mean").sort_index()
            df = df.shift(1)

        # Mode 2) Smoothed prices (exponentially weighted moving average)
        # ewm().mean() for seasonal group t uses groups 0..t; shift(1) makes group t use groups 0..t-1 only — causal.
        elif operating_protocol_transform == "Smoothed":
            df = pd.pivot_table(price_df,values="Price",index=["Year", "Seasonal_group"],columns="Interval",aggfunc="mean").sort_index()
            df = df.ewm(alpha=0.1).mean().shift(1)

        # Mode 3) Pessimistic prices (5th percentile)
        # shift(1) so each seasonal group's signal is derived from the previous seasonal group's quantile.
        elif operating_protocol_transform == "Pessimistic":
            df = pd.pivot_table(price_df,values='Price',index=["Year", "Seasonal_group"],columns="Interval",aggfunc=lambda x: x.quantile(0.05)).sort_index()
            df = df.shift(1)

        # Mode 4) Pessimistic smoothed prices
        elif operating_protocol_transform == "Pessimistic_Smoothed":
            df = pd.pivot_table(price_df,values='Price',index=["Year", "Seasonal_group"],columns="Interval",aggfunc=lambda x: x.quantile(0.05)).sort_index()
            df = df.ewm(alpha=0.1).mean().shift(1)

        return df

    def _apply_blend(operating_protocol_window_size):
        # Use a separate original copy so that updating interval i does not cascade into interval i+1.
        # Use a backward-looking window [i, i-1, i-2, ...] so interval i is never smoothed using
        # a future interval's price.
        original = transformed_df.copy()
        df = transformed_df.copy()
        for i, protocol_interval in enumerate(df.columns):
            sliding_window_of_x_protocol_intervals = [
                original.columns[
                    (i - offset) % len(original.columns)
                    ] for offset in range(int(operating_protocol_window_size))
                ]
            df[protocol_interval] = original[sliding_window_of_x_protocol_intervals].mean(axis=1) 
        
        return df
        
     
    
    def _create_predicted_price():
        df = (transformed_df.stack().rename("Predicted_price").reset_index())
        df = price_df.copy().merge(df.copy(), on=["Year", "Seasonal_group", "Interval"], how="left")
        df = df[['Date','Predicted_price']]
        return df
    

    def _create_direct_bess_instructions():
       
        def _translate_price_to_rank():
            # No additional shift(1) here — all transforms now apply shift(1) themselves,
            # so transformed_df for seasonal group t already reflects the previous seasonal group's data.
            df = (transformed_df.rank(axis=1, method="first").dropna(how="any").astype(int))
            df = (df.stack().rename("Rank").reset_index())
            df.columns = ["Year", "Seasonal_group", "Interval", "Rank"]

            return df

        def _translate_rank_to_bess_instruction(rank_df):
            number_of_intervals_to_instruct = int(bess_hours * 60 / variables.HORIZON_GRANULARITY_IN_MINUTES)
            n_intervals = len(transformed_df.columns)
            df = price_df.copy().merge(rank_df, on=["Year", "Seasonal_group", "Interval"], how="left")
            df["BESS_instruction"] = 0
            df.loc[df["Rank"] <= number_of_intervals_to_instruct, "BESS_instruction"] = 1 # Discharge 
            df.loc[df["Rank"] > n_intervals - number_of_intervals_to_instruct, "BESS_instruction"] = 2 # Charge
            df = df[['Date','BESS_instruction']]

            return df
        
        rank_df = _translate_price_to_rank()
        df = _translate_rank_to_bess_instruction(rank_df)

        return df

    def _filter():
        df = Predicted_price.merge(BESS_instruction, on="Date", how="left")
        df = df.loc[(df["Date"] >= variables.PIPELINE_START_DATE)& (df["Date"] <= variables.PIPELINE_END_DATE)].copy()
        return df.reset_index(drop=True)

    
    if apply_blend:
        transformed_df = _apply_transform(operating_protocol_transform)
        transformed_df = _apply_blend(operating_protocol_window_size)
    else:
        transformed_df = _apply_transform(operating_protocol_transform)
    

    Predicted_price = _create_predicted_price()
    BESS_instruction = _create_direct_bess_instructions()
    heuristic_predicted_price_df = _filter()

    return heuristic_predicted_price_df


Comparing multiple

In [3]:
if False:
        operating_protocol_seasonal_grouping_granularity = ["4D"]
        operating_protocol_transform = ["Smoothed"]
        operating_protocol_window_size = [2]
        apply_blend = [True,False]

        items = [
                (a, b, c, d)
                for a in operating_protocol_seasonal_grouping_granularity
                for b in operating_protocol_transform
                for c in operating_protocol_window_size
                for d in apply_blend
        ]

        actual_price = pd.read_csv("../1_Dataset/2_Processed_data/price.csv")
        actual_price["Date"] = pd.to_datetime(actual_price["Date"])

        comp_dict = {"actual_price": actual_price}

        for a, b, c, d in items:
                p = heuristic_price_prediction(
                        price_df_path = "../1_Dataset/2_Processed_data/price_extra_14_days.csv",
                        operating_protocol_seasonal_grouping_granularity = a,
                        operating_protocol_transform = b,
                        operating_protocol_window_size = c,
                        apply_blend = d
                )
        key = f"{a}|{b}|w{c}|blend={d}"
        comp_dict[key] = p




        from IPython.display import HTML

        df_combined = pd.concat([df.set_index("Date").iloc[:, 0].rename(name) for name, df in comp_dict.items()], axis=1)
        styled_df = df_combined.head(100).style.background_gradient(cmap="RdYlGn", axis=None).format("{:.2f}")
        styled_df.to_excel("Matrix.xlsx")
        HTML(styled_df.to_html())


In [4]:
# Heuristic operating-protocol settings.
bess_hours = 2.0
operating_protocol_seasonal_grouping_granularity = "14D"
operating_protocol_transform = "None"
operating_protocol_window_size = 2
apply_blend = False

In [5]:
def _data_prep(price_df, operating_protocol_seasonal_grouping_granularity):
        df = price_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df["Year"] = df["Date"].dt.year
        df["Seasonal_group"] = (df.groupby(pd.Grouper(key="Date", freq=operating_protocol_seasonal_grouping_granularity)).ngroup() + 1)
        # Dispatch timestamps are interval-ending. Move each 5-minute observation
        # back one step before assigning it to a 30-minute operating interval.
        interval_start = df["Date"] - pd.Timedelta(minutes=variables.FEATURE_GRANULARITY_IN_MINUTES)
        df["Interval"] = (interval_start.dt.hour * (60 // int(variables.HORIZON_GRANULARITY_IN_MINUTES)) + interval_start.dt.minute // int(variables.HORIZON_GRANULARITY_IN_MINUTES)+ 1)
        
        return df


# Call the heuristic creation function & create the predictions (from 1 price curve).
# Keep the original function and settings above; only the dataset adapter is new.
raw_price = (
        pd.read_parquet(variables.TARGET_DATASET_PATH, columns=[variables.SELECTED_TARGET_COLUMN_NAME])
        .rename(columns={variables.SELECTED_TARGET_COLUMN_NAME: "Price"})
        .reset_index()
        .rename(columns={"SETTLEMENTDATE": "Date", "index": "Date"})
)
price_df = _data_prep(raw_price, operating_protocol_seasonal_grouping_granularity)

# Call function
predicted_price = heuristic_price_prediction(
        price_df = price_df,
        operating_protocol_transform = operating_protocol_transform,
        operating_protocol_window_size = operating_protocol_window_size,
        apply_blend = apply_blend
)
# Export the original single-timeline result as a useful diagnostic.
variables.MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
predicted_price.to_csv(variables.MODEL_RESULTS_DIR / "operating_protocol_timeline.csv",index=False)


## Complete forecast archive

Freeze the operating profile available at each forecast origin and save all 96 future half-hour predictions. Freezing the profile is important: a forecast that crosses into the next seasonal group must not use observations collected after its origin.

In [6]:
HORIZON_LIST = range(1, variables.HORIZON_COUNT + 1)
target_columns = [f"target_h{horizon}" for horizon in HORIZON_LIST]
forecast_origins = (
        pd.read_parquet(variables.AGG_TARGET_DATASET_PATH, columns=target_columns)
        .dropna(subset=target_columns)
        .index
)

# Recover the transformed daily operating profile for every seasonal group.
protocol_rows = price_df[["Date", "Year", "Seasonal_group", "Interval"]].merge(
        predicted_price[["Date", "Predicted_price"]], on="Date", how="inner"
)
protocol_profiles = pd.pivot_table(
        protocol_rows, values="Predicted_price",
        index=["Year", "Seasonal_group"], columns="Interval", aggfunc="mean"
).sort_index().reindex(columns=range(1, 24 * 60 // variables.HORIZON_GRANULARITY_IN_MINUTES + 1))

origin_metadata = (
        pd.DataFrame({"Date": forecast_origins})
        .merge(price_df[["Date", "Year", "Seasonal_group"]].drop_duplicates("Date"), on="Date", how="left")
)
origin_groups = pd.MultiIndex.from_frame(origin_metadata[["Year", "Seasonal_group"]])
profile_row = protocol_profiles.index.get_indexer(origin_groups)
profile_values = protocol_profiles.to_numpy(dtype=np.float32)
profile_values = np.vstack([profile_values, np.full((1, profile_values.shape[1]), np.nan, dtype=np.float32)])
profile_row[profile_row < 0] = len(profile_values) - 1

# Each target is the mean of six future 5-minute RRPs. Apply the frozen
# operating profile to those same six constituent timestamps, which also
# handles forecast origins that are not aligned to a wall-clock half-hour.
forecast_matrix = np.empty((len(forecast_origins), variables.HORIZON_COUNT), dtype=np.float32)
for column, horizon in enumerate(tqdm(HORIZON_LIST, desc="operating-protocol horizons")):
        component_intervals = []
        for step in range(1, variables.FEATURE_STEPS_PER_HORIZON + 1):
                component_time = forecast_origins + pd.Timedelta(
                        minutes=(horizon - 1) * variables.HORIZON_GRANULARITY_IN_MINUTES
                        + step * variables.FEATURE_GRANULARITY_IN_MINUTES
                )
                interval_start = component_time - pd.Timedelta(minutes=variables.FEATURE_GRANULARITY_IN_MINUTES)
                interval_number = (
                        interval_start.hour * (60 // variables.HORIZON_GRANULARITY_IN_MINUTES)
                        + interval_start.minute // variables.HORIZON_GRANULARITY_IN_MINUTES
                ).astype(int)
                component_intervals.append(profile_values[profile_row, interval_number])
        forecast_matrix[:, column] = np.mean(component_intervals, axis=0, dtype=np.float32)

operating_protocol_forecasts = pd.DataFrame(
        forecast_matrix, index=forecast_origins,
        columns=[f"predicted_h{horizon}" for horizon in HORIZON_LIST],
)
operating_protocol_forecasts.index.name = "forecast_origin"
operating_protocol_forecasts.to_parquet(variables.OPERATING_PROTOCOL_FORECASTS_PATH)
print(f"Saved {len(operating_protocol_forecasts):,} complete operating-protocol forecast paths to {variables.OPERATING_PROTOCOL_FORECASTS_PATH}")

operating-protocol horizons: 100%|██████████| 96/96 [00:20<00:00,  4.60it/s]


Saved 736,417 complete operating-protocol forecast paths to C:\Users\Daniel\OneDrive\Documents\2_Energy_toolkit\1_ST_price_prediction\EPF\5_Model\Data\5_model_results\operating_protocol_forecasts.parquet
